## 라이브러리 호출 및 DPU 환경 설정

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
from pynq import Overlay
from pynq import MMIO
import time
import keyboard
from pynq_dpu import DpuOverlay

import numpy as np
import cv2
import os
import sys
from IPython.display import display, clear_output
import ipywidgets as widgets
import spidev


# DPU bit인지 아닌지 
use_dpu = False    

if use_dpu:
    DPU_path = "../dpu/dpu.bit"
    xmodel_path = "../xmodel/top-tiny-yolov3_coco_256.xmodel"
    
    overlay = DpuOverlay(DPU_path)
    overlay.load_model(xmodel_path)
    
else:
    overlay = Overlay('./HW_setting_test_bit/dpu.bit')

## 모터 초기설정

In [2]:
# 모터 설정
motor_0_address = 0x00A0000000
motor_1_address = 0x00A0010000
motor_2_address = 0x00A0020000
motor_3_address = 0x00A0030000
motor_4_address = 0x00A0040000
motor_5_address = 0x00A0050000

address_range = 0x10000

motor_0 = MMIO(motor_0_address , address_range)
motor_1 = MMIO(motor_1_address , address_range) # CHANGE 
motor_2 = MMIO(motor_2_address , address_range)
motor_3 = MMIO(motor_3_address , address_range)
motor_4 = MMIO(motor_4_address , address_range) # CHANGE
motor_5 = MMIO(motor_5_address , address_range)

In [3]:
# 모터 기본 설정
# size = 2ms ==3.33ns x 600600.600..... 
# about 600600
size = 600600
if use_dpu:
    size = 600600 #300MHz 기준   
else:
    size = 200000 #100MHz 기준

motor_0_percent = 100/100
motor_1_percent = 100/100
motor_2_percent = 100/100
motor_3_percent = 100/100
motor_4_percent = 100/100
motor_5_percent = 100/100

motor_0_duty = size * motor_0_percent
motor_1_duty = size * motor_1_percent
motor_2_duty = size * motor_2_percent
motor_3_duty = size * motor_3_percent
motor_4_duty = size * motor_4_percent
motor_5_duty = size * motor_5_percent
print(int(motor_0_duty))


motor_0.write(0x00,size)           # size
motor_0.write(0x04,int(motor_0_duty))   # DUTY
motor_0.write(0x08,0)              # valid

motor_1.write(0x00,size)  # size
motor_1.write(0x04,int(motor_1_duty))   # DUTY
motor_1.write(0x08,0)       # valid

motor_2.write(0x00,size)  # size
motor_2.write(0x04,int(motor_2_duty))   # DUTY
motor_2.write(0x08,0)       # valid

motor_3.write(0x00,size)  # size
motor_3.write(0x04,int(motor_3_duty))   # DUTY
motor_3.write(0x08,0)       # valid

motor_4.write(0x00,size)  # size
motor_4.write(0x04,int(motor_4_duty))   # DUTY
motor_4.write(0x08,0)       # valid

motor_5.write(0x00,size)  # size
motor_5.write(0x04,int(motor_5_duty))   # DUTY
motor_5.write(0x08,0)       # valid

200000


## 모터 테스트

In [11]:
# motor 4  rear_right_front
motor_0.write(0x08,0)          
motor_1.write(0x08,0)         

motor_2.write(0x08,0)       
motor_3.write(0x08,0)            

motor_4.write(0x08,1)           
motor_5.write(0x08,0)  

In [12]:
# motor 0 rear_right_back
motor_0.write(0x08,1)             
motor_1.write(0x08,0)              

motor_2.write(0x08,0)           
motor_3.write(0x08,0)         

motor_4.write(0x08,0)             
motor_5.write(0x08,0)             

In [13]:
# motor 3 rear_left_front
motor_0.write(0x08,0)            
motor_1.write(0x08,0)            

motor_2.write(0x08,0)           
motor_3.write(0x08,1)     

motor_4.write(0x08,0)           
motor_5.write(0x08,0)     

In [14]:
# motor 2 rear_left_back
motor_0.write(0x08,0)              
motor_1.write(0x08,0)          

motor_2.write(0x08,1)             
motor_3.write(0x08,0)             

motor_4.write(0x08,0)          
motor_5.write(0x08,0)    

In [17]:
# motor 1 steering_left
motor_0.write(0x08,0)             
motor_1.write(0x08,1)            

motor_2.write(0x08,0)              
motor_3.write(0x08,0)          

motor_4.write(0x08,0)              
motor_5.write(0x08,0)              

In [18]:
# motor 5 steering_right 
motor_0.write(0x08,0)              
motor_1.write(0x08,0)           

motor_2.write(0x08,0)           
motor_3.write(0x08,0)           

motor_4.write(0x08,0)        
motor_5.write(0x08,1)      

In [19]:
# OFF
motor_5.write(0x08,0)            # valid  steering_right
motor_1.write(0x08,0)             # valid  steering_left
motor_2.write(0x08,0)              # valid rear_left_back
motor_3.write(0x08,0)              # valid rear_left_front
motor_4.write(0x08,0)              # valid rear_right_front
motor_0.write(0x08,0)              # valid rear_right_back

## SPI TEST

In [19]:
# 디스플레이 확인 후 인터럽트해야 다음 코드 실행 가능
# 4자리에서 3자리 전환 시 이전 값이 일부 남아 숫자가 겹쳐 보일 수 있음
# 예: 1023 다음 900 출력 시 900이 아닌 9003처럼 표시될 수 있음

import spidev
import time
import sys

# SPI 설정
spi0 = spidev.SpiDev()

spi0.open(0, 0) 

# SPI 속도 설정 (최대 20MHz, Pmod AD1은 최대 1MHz 권장)
spi0.max_speed_hz = 20000000

# SPI 모드 설정 (모드 0)
spi0.mode = 0b00

def read_adc(spi):
    adc_response = spi.xfer2([0x00, 0x00])
    adc_value = ((adc_response[0] & 0x0F) << 8) | adc_response[1]
    return adc_value

try:
    while True:
        # D0 채널에서 데이터 읽기
        adc_value0 = read_adc(spi0)
        # 출력 내용을 덮어쓰기
        #sys.stdout.write(f"\rD0 채널 값: {adc_value0*3.3/4096}")
        sys.stdout.write(f"\rD0 채널 값: {adc_value0}")
        sys.stdout.flush()
        time.sleep(0.5)
except KeyboardInterrupt:
    spi0.close()
##우측 240
#좌측 990

D0 채널 값: 1808

# 카메라 출력 확인

In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("카메라를 열 수 없습니다.")
else:
    image_widget = widgets.Image(format='jpeg')
    display(image_widget)

    try:
        while True:
            ret, frame = cap.read()

            if not ret:
                print("프레임을 읽을 수 없습니다.")
                break

            ret, jpeg = cv2.imencode('.jpeg', frame)
            if not ret:
                print("JPEG 인코딩에 실패했습니다.")
                break

            image_widget.value = jpeg.tobytes()

    except KeyboardInterrupt:
        print("중지되었습니다.")
    finally:
        cap.release()

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (616) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
[ WARN:0] global ./modules/videoio/src/cap_v4l.cpp (890) open VIDEOIO(V4L2:/dev/video0): can't open camera by index
